# Can room occupancy be predicted from environmental sensors?

This notebook tests whether `Occupancy` can be predicted from temperature, humidity, light, CO₂, and humidity ratio. Because the rows are time ordered and adjacent measurements are strongly related, the final 30% of observations is held out as a **future test period**. This is a stricter and more realistic test than randomly shuffling individual rows.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay,
    accuracy_score, average_precision_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

## 1. Load and validate the data

In [ ]:
DATA_PATH = Path("data.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Expected {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
feature_cols = ["Temperature", "Humidity", "Light", "CO2", "HumidityRatio"]
target_col = "Occupancy"

assert set(df[target_col].unique()).issubset({0, 1}), "Occupancy must be binary"
assert df["date"].is_monotonic_increasing, "Rows must be chronological"

quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique(),
})
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Period: {df['date'].min()} to {df['date'].max()}")
print(f"Duplicate rows: {df.duplicated().sum():,}")
display(quality)
display(df.head())

In [ ]:
class_summary = (
    df[target_col].value_counts().sort_index().rename("count").to_frame()
    .assign(percent=lambda x: 100 * x["count"] / len(df))
)
class_summary.index = class_summary.index.map({0: "Unoccupied", 1: "Occupied"})
display(class_summary.style.format({"count": "{:,}", "percent": "{:.1f}%"}))

ax = sns.countplot(data=df, x=target_col, hue=target_col, legend=False, palette="Set2")
ax.set(title="Target class balance", xlabel="Occupancy (0 = no, 1 = yes)", ylabel="Rows")
plt.show()

## 2. Exploratory analysis

The distributions below show how each sensor differs between occupied and unoccupied periods. They are descriptive only; the untouched future test period is not used to fit the models.

In [ ]:
display(df.groupby(target_col)[feature_cols].agg(["mean", "median", "std"]).round(4))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, feature_cols):
    sns.boxplot(data=df, x=target_col, y=col, ax=ax, showfliers=False, hue=target_col, legend=False, palette="Set2")
    ax.set_title(col)
axes.flat[-1].axis("off")
fig.suptitle("Sensor distributions by occupancy", y=1.02, fontsize=15)
fig.tight_layout()
plt.show()

## 3. Chronological train/test split

The first 70% is used for model development; the last 30% is held out for one final evaluation. `date` is retained for splitting and reporting but is deliberately excluded from the predictors. A majority-class dummy model establishes the baseline.

In [ ]:
split_idx = int(len(df) * 0.70)
train = df.iloc[:split_idx].copy()
test = df.iloc[split_idx:].copy()
X_train, y_train = train[feature_cols], train[target_col]
X_test, y_test = test[feature_cols], test[target_col]

split_summary = pd.DataFrame({
    "rows": [len(train), len(test)],
    "start": [train.date.min(), test.date.min()],
    "end": [train.date.max(), test.date.max()],
    "occupied_%": [100 * y_train.mean(), 100 * y_test.mean()],
}, index=["Train", "Test"])
display(split_summary.style.format({"rows": "{:,}", "occupied_%": "{:.1f}%"}))

## 4. Compare candidate models

The linear model is scaled and class-weighted. Tree models capture nonlinear relationships. Balanced accuracy and F1 are emphasized because plain accuracy can look good by always predicting the majority class. Model selection uses expanding-window time-series cross-validation on the training period only.

In [ ]:
models = {
    "Majority baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2_000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "Random forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300, min_samples_leaf=5, class_weight="balanced",
            n_jobs=1, random_state=RANDOM_STATE
        )),
    ]),
    "Histogram gradient boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(max_iter=200, l2_regularization=1, random_state=RANDOM_STATE)),
    ]),
}

tscv = TimeSeriesSplit(n_splits=5)
scoring = {"balanced_accuracy": "balanced_accuracy", "f1": "f1", "roc_auc": "roc_auc"}
cv_rows = []
for name, model in models.items():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        scores = cross_validate(model, X_train, y_train, cv=tscv, scoring=scoring)
    cv_rows.append({
        "model": name,
        **{f"mean_{metric}": np.nanmean(scores[f'test_{metric}']) for metric in scoring},
        **{f"std_{metric}": np.nanstd(scores[f'test_{metric}']) for metric in scoring},
    })
cv_results = pd.DataFrame(cv_rows).set_index("model").sort_values("mean_f1", ascending=False)
display(cv_results.style.format("{:.3f}").background_gradient(subset=["mean_balanced_accuracy", "mean_f1", "mean_roc_auc"], cmap="Greens"))

## 5. Final evaluation on the untouched future period

The model with the best mean cross-validated training F1 is selected **before** inspecting the holdout. All candidate models are then fit on the full training period and evaluated once on the holdout, with balanced accuracy and probability-ranking metrics shown for context.

In [ ]:
test_rows = []
fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    test_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, proba),
        "average_precision": average_precision_score(y_test, proba),
    })
test_results = pd.DataFrame(test_rows).set_index("model").sort_values("f1", ascending=False)
display(test_results.style.format("{:.3f}").background_gradient(cmap="Greens"))

best_name = cv_results.index[0]  # selected using training-period CV, not test performance
best_model = fitted_models[best_name]
best_pred = best_model.predict(X_test)
best_proba = best_model.predict_proba(X_test)[:, 1]
print(f"Best holdout F1: {best_name} ({test_results.loc[best_name, 'f1']:.3f})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, best_pred, display_labels=["Unoccupied", "Occupied"], cmap="Blues", ax=axes[0], colorbar=False)
axes[0].set_title(f"Confusion matrix: {best_name}")
RocCurveDisplay.from_predictions(y_test, best_proba, ax=axes[1], name=best_name)
axes[1].plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
axes[1].set_title("ROC curve")
PrecisionRecallDisplay.from_predictions(y_test, best_proba, ax=axes[2], name=best_name)
axes[2].axhline(y_test.mean(), linestyle="--", color="gray", linewidth=1, label="Class prevalence")
axes[2].set_title("Precision–recall curve")
fig.tight_layout()
plt.show()

print(classification_report(y_test, best_pred, target_names=["Unoccupied", "Occupied"], digits=3))

## 6. What drives the predictions?

Permutation importance measures the drop in holdout F1 after shuffling one feature. It works consistently across all model types and describes predictive usefulness—not causality.

In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test, scoring="f1", n_repeats=10,
    random_state=RANDOM_STATE, n_jobs=1,
)
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=True)

ax = importance.plot.barh(
    x="feature", y="importance_mean", xerr="importance_std",
    legend=False, figsize=(8, 4), color="#4C78A8", capsize=3,
)
ax.set(title=f"Permutation importance for {best_name}", xlabel="Mean decrease in test F1 after shuffling", ylabel="")
plt.tight_layout()
plt.show()
display(importance.sort_values("importance_mean", ascending=False).reset_index(drop=True).style.format({"importance_mean": "{:.4f}", "importance_std": "{:.4f}"}))

## 7. Conclusion and limitations

In [ ]:
best = test_results.loc[best_name]
tn, fp, fn, tp = confusion_matrix(y_test, best_pred).ravel()
print("YES — occupancy is highly predictable from these sensor readings on the future holdout period.")
print(
    f"{best_name} achieved {best['accuracy']:.1%} accuracy, "
    f"{best['balanced_accuracy']:.1%} balanced accuracy, {best['f1']:.1%} F1, "
    f"and {best['roc_auc']:.1%} ROC AUC."
)
print(f"It found {tp:,} of {tp + fn:,} occupied rows, with {fp:,} false occupied alarms.")
print("\nCaveats:")
print("• These rows cover one room and a short February 2015 period; performance may drift in another room, season, or sensor setup.")
print("• Adjacent minute readings are autocorrelated. The future holdout reduces leakage, but a multi-site or multi-season external test would be stronger.")
print("• Strong reliance on Light may partly encode occupant behavior (for example, switching lights on). If lights can remain on when empty, validate under that condition.")
print("• For deployment, choose a probability threshold based on the relative cost of missed occupancy versus false alarms, then monitor drift.")